# 1943 — Warren McCulloch & Walter Pitts
## Artificial Neuron (เซลล์ประสาทเทียมตัวแรกของโลก)

| | |
|---|---|
| **ผู้คิดค้น** | Warren McCulloch (นักประสาทสรีรวิทยา) และ Walter Pitts (นักตรรกศาสตร์) |
| **ผลงาน** | *"A Logical Calculus of the Ideas Immanent in Nervous Activity"* — Bulletin of Mathematical Biophysics, 1943 |
| **แนวคิดหลัก** | เซลล์ประสาทหนึ่งตัวทำงานเหมือน **ประตูตรรกะ (logic gate)** — รับสัญญาณ 0/1 แล้วตัดสินใจ "ยิง" (1) หรือ "ไม่ยิง" (0) |
| **ความสำคัญ** | เป็นครั้งแรกที่มีคนแสดงว่า **เครือข่ายของเซลล์ประสาทแบบง่ายๆ คำนวณฟังก์ชันตรรกะได้ทุกแบบ** — จุดเริ่มต้นของ Neural Network ทั้งหมด |

> 🧠 **ไอเดีย:** สมองอาจเป็น "เครื่องคำนวณตรรกะ" ที่สร้างจากหน่วยเล็กๆ ที่ทำงานแบบ ใช่/ไม่ใช่

## 1. แบบจำลอง McCulloch–Pitts (MP Neuron)

เซลล์ประสาทมีอินพุต 2 ประเภท:
- **Excitatory (กระตุ้น)** $x_i \in \{0,1\}$ — ช่วยให้ยิง
- **Inhibitory (ยับยั้ง)** $x_j \in \{0,1\}$ — ถ้ามีตัวใดตัวหนึ่งเป็น 1 จะ **ห้ามยิงทันที** (absolute inhibition ตามบทความต้นฉบับ)

และมี **threshold** $\theta$ ที่ผู้ออกแบบกำหนดเอง

$$
y = \begin{cases}
1 & \text{ถ้า } \displaystyle\sum_{i \in \text{excitatory}} x_i \ge \theta \;\text{ และไม่มีอินพุตยับยั้งตัวใดเป็น 1} \\
0 & \text{กรณีอื่น}
\end{cases}
$$

**สิ่งที่ต้องจำ:** MP neuron **ไม่มีการเรียนรู้** — ค่า $\theta$ และการต่อสาย ต้องออกแบบด้วยมือทั้งหมด (การเรียนรู้จากข้อมูลมาในปี 1958 กับ Perceptron)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product


def mp_neuron(excitatory, inhibitory, theta):
    if any(inhibitory):
        return 0
    return int(sum(excitatory) >= theta)


def truth_table(fn, n_inputs):
    names = [f"x{i+1}" for i in range(n_inputs)]
    rows = [{**dict(zip(names, xs)), "y": fn(*xs)} for xs in product([0, 1], repeat=n_inputs)]
    return pd.DataFrame(rows)

## 2. ตัวอย่างการคำนวณด้วยมือ: AND, OR, NOT

### AND — ใช้ $\theta = 2$ (ต้องมีอินพุตเป็น 1 ครบทั้ง 2 ตัว)

| $x_1$ | $x_2$ | ผลรวม $x_1+x_2$ | $\ge 2$ ? | $y$ |
|:-:|:-:|:-:|:-:|:-:|
| 0 | 0 | 0 | ไม่ | **0** |
| 0 | 1 | 1 | ไม่ | **0** |
| 1 | 0 | 1 | ไม่ | **0** |
| 1 | 1 | 2 | ใช่ | **1** |

### OR — ใช้ $\theta = 1$ (มีอินพุตเป็น 1 อย่างน้อย 1 ตัวก็พอ)

| $x_1$ | $x_2$ | ผลรวม | $\ge 1$ ? | $y$ |
|:-:|:-:|:-:|:-:|:-:|
| 0 | 0 | 0 | ไม่ | **0** |
| 0 | 1 | 1 | ใช่ | **1** |
| 1 | 0 | 1 | ใช่ | **1** |
| 1 | 1 | 2 | ใช่ | **1** |

### NOT — อินพุตเป็นแบบ **ยับยั้ง** และ $\theta = 0$

- $x = 0$: ไม่มีการยับยั้ง, ผลรวม excitatory $= 0 \ge 0$ → $y = 1$
- $x = 1$: ถูกยับยั้ง → $y = 0$

> สังเกต: **gate เดียวกัน เปลี่ยนแค่ $\theta$** ก็เปลี่ยนจาก AND เป็น OR ได้ — threshold คือ "ความเข้มงวด" ของเซลล์ประสาท

มาตรวจคำตอบด้วยโค้ด:

In [ ]:
AND = lambda x1, x2: mp_neuron([x1, x2], [], theta=2)
OR = lambda x1, x2: mp_neuron([x1, x2], [], theta=1)
NOT = lambda x: mp_neuron([], [x], theta=0)

for name, fn, n in [("AND (theta=2)", AND, 2), ("OR (theta=1)", OR, 2), ("NOT (inhibitory, theta=0)", NOT, 1)]:
    print(f"=== {name} ===")
    print(truth_table(fn, n).to_string(index=False), "\n")

print("=== แสดงขั้นตอนคำนวณ AND ทีละแถว ===")
for x1, x2 in product([0, 1], repeat=2):
    s = x1 + x2
    print(f"x=({x1},{x2})  sum={s}  sum>=2? {str(s >= 2):5}  ->  y={AND(x1, x2)}")

## 3. มองแบบเรขาคณิต: threshold = เส้นตรงแบ่งระนาบ

เงื่อนไข $x_1 + x_2 \ge \theta$ คือ **พื้นที่เหนือเส้นตรง** $x_1 + x_2 = \theta$
- AND: เส้น $x_1 + x_2 = 2$ → มีแค่จุด (1,1) ที่อยู่ฝั่ง "ยิง"
- OR: เส้น $x_1 + x_2 = 1$ → จุด (0,1), (1,0), (1,1) อยู่ฝั่ง "ยิง"

ภาพนี้สำคัญมาก เพราะมันอธิบายว่า **ทำไมเซลล์ประสาทตัวเดียวถึงแก้ XOR ไม่ได้** (ดูหัวข้อถัดไป)

In [ ]:
pts = list(product([0, 1], repeat=2))
xs = np.linspace(-0.5, 1.5, 100)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, (name, fn, theta) in zip(axes, [("AND", AND, 2), ("OR", OR, 1)]):
    ax.fill_between(xs, theta - xs, 3, color="tab:green", alpha=0.15, label="fires: x1 + x2 >= theta")
    ax.plot(xs, theta - xs, "k--", label=f"x1 + x2 = {theta}")
    for a, b in pts:
        y = fn(a, b)
        ax.scatter(a, b, s=220, c="tab:green" if y else "tab:red", edgecolors="k", zorder=3)
        ax.annotate(f"y={y}", (a, b), textcoords="offset points", xytext=(12, 8))
    ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x1", ylabel="x2",
           title=f"McCulloch-Pitts {name} (theta={theta})")
    ax.set_aspect("equal")
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## 4. XOR: เซลล์ตัวเดียวทำไม่ได้ — แต่ "เครือข่าย" ทำได้

XOR ให้ผล 1 เมื่ออินพุต **ต่างกัน**: (0,1)→1, (1,0)→1 แต่ (0,0)→0, (1,1)→0

ไม่มีเส้นตรงเส้นเดียวที่แยกจุดสีเขียว {(0,1),(1,0)} ออกจากจุดสีแดง {(0,0),(1,1)} ได้
เราพิสูจน์ด้วยการ **ลองทุกแบบ**: ให้แต่ละอินพุตเป็น excitatory / inhibitory / ไม่ต่อ และลอง $\theta = 0,1,2,3$

In [ ]:
XOR_TARGET = {(0, 0): 0, (0, 1): 1, (1, 0): 1, (1, 1): 0}
roles = ["excitatory", "inhibitory", "unused"]

tried, found = 0, []
for r1, r2 in product(roles, repeat=2):
    for theta in range(4):
        tried += 1

        def neuron(x1, x2):
            exc = [x for x, r in [(x1, r1), (x2, r2)] if r == "excitatory"]
            inh = [x for x, r in [(x1, r1), (x2, r2)] if r == "inhibitory"]
            return mp_neuron(exc, inh, theta)

        if all(neuron(*x) == y for x, y in XOR_TARGET.items()):
            found.append((r1, r2, theta))

print(f"ลองทั้งหมด {tried} แบบ  ->  แบบที่คำนวณ XOR ได้: {len(found)} แบบ")

### สร้าง XOR จากเซลล์ 3 ตัว

แนวคิด: $\text{XOR} = (x_1 \text{ OR } x_2) \text{ AND NOT } (x_1 \text{ AND } x_2)$

```
x1 ──┬──► [h1: OR,  θ=1] ──(excitatory)──┐
     │                                    ├──► [y: θ=1] ──► XOR
x2 ──┴──► [h2: AND, θ=2] ──(inhibitory)──┘
```

**คำนวณด้วยมือ** ทุกกรณี:

| $x_1$ | $x_2$ | $h_1$ = OR | $h_2$ = AND | $y$: $h_1 \ge 1$ และ $h_2 = 0$ ? |
|:-:|:-:|:-:|:-:|:-:|
| 0 | 0 | 0 | 0 | $0 \ge 1$ ไม่ → **0** |
| 0 | 1 | 1 | 0 | $1 \ge 1$ และไม่ถูกยับยั้ง → **1** |
| 1 | 0 | 1 | 0 | $1 \ge 1$ และไม่ถูกยับยั้ง → **1** |
| 1 | 1 | 1 | 1 | ถูกยับยั้งโดย $h_2$ → **0** |

นี่คือบทเรียนใหญ่ของปี 1943: **เซลล์ตัวเดียวมีขีดจำกัด แต่เมื่อต่อกันเป็นชั้นๆ จะคำนวณอะไรก็ได้**
(ปัญหาที่เหลือคือ — จะหาค่า weight/threshold ของเครือข่ายใหญ่ๆ "อัตโนมัติ" ได้อย่างไร?)

In [ ]:
def XOR(x1, x2, show=False):
    h1 = mp_neuron([x1, x2], [], theta=1)
    h2 = mp_neuron([x1, x2], [], theta=2)
    y = mp_neuron([h1], [h2], theta=1)
    if show:
        print(f"x=({x1},{x2})  h1(OR)={h1}  h2(AND)={h2}  ->  y={y}")
    return y


for x1, x2 in product([0, 1], repeat=2):
    XOR(x1, x2, show=True)

assert all(XOR(*x) == y for x, y in XOR_TARGET.items())
print("\nเครือข่าย 3 เซลล์คำนวณ XOR ถูกทุกกรณี")

## 5. แบบฝึกหัด

1. ออกแบบ MP neuron ที่ให้ผล **"เสียงข้างมาก" (Majority) ของ 3 อินพุต** — ยิงเมื่อมีอินพุตเป็น 1 อย่างน้อย 2 ตัว ต้องใช้ $\theta$ เท่าไร?
2. ออกแบบ $y = x_1 \text{ AND NOT } x_2$ ด้วยเซลล์ **ตัวเดียว** (ใบ้: ใช้อินพุตยับยั้ง)
3. เครือข่ายที่ใช้ทำ XOR ด้านบนต้องใช้กี่เซลล์? ลองคิดว่า XNOR (ผลตรงข้ามกับ XOR) ต้องแก้ตรงไหน

รันเซลล์ถัดไปเพื่อดูเฉลยข้อ 1–2

In [ ]:
MAJ3 = lambda x1, x2, x3: mp_neuron([x1, x2, x3], [], theta=2)
AND_NOT = lambda x1, x2: mp_neuron([x1], [x2], theta=1)

print("เฉลยข้อ 1: Majority of 3  (theta = 2)")
print(truth_table(MAJ3, 3).to_string(index=False))
print("\nเฉลยข้อ 2: x1 AND NOT x2  (x1 = excitatory, x2 = inhibitory, theta = 1)")
print(truth_table(AND_NOT, 2).to_string(index=False))

## 6. สรุป

| จุดแข็ง | ข้อจำกัด |
|---|---|
| โมเดลคณิตศาสตร์ของเซลล์ประสาทตัวแรก | อินพุต/เอาต์พุตเป็น 0/1 เท่านั้น |
| แสดงว่าเครือข่ายเซลล์คำนวณตรรกะได้ทุกแบบ | **เรียนรู้ไม่ได้** ต้องออกแบบค่าด้วยมือ |
| เป็นรากฐานของคอมพิวเตอร์ยุคแรก (von Neumann อ้างถึง) | ทุกอินพุตมีน้ำหนักเท่ากัน (ไม่มี weight จริงจัง) |

**เส้นทางต่อไป:** 1949 Donald Hebb เสนอว่าเซลล์ที่ยิงพร้อมกัน "การเชื่อมต่อจะแข็งแรงขึ้น" → 1958 Frank Rosenblatt นำแนวคิดนี้มาสร้าง **Perceptron ที่เรียนรู้ weight เองได้** (ดูไฟล์ `1958.ipynb`)